# 02 · Línea base y chequeos previos al entrenamiento

1. **LB-0**: predictor que siempre emite la media de train (umbral de MAE a superar).
2. Chequeos por arquitectura: MSE inicial ≈ 1, sobreajuste de 64 ejemplos, gradientes finitos y no nulos, `eval()` determinista.

Solo se usan train y val (test no se toca).

In [1]:
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "bp").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, torch
print("ROOT:", ROOT, "| cuda:", torch.cuda.is_available())

ROOT: C:\Users\USUARIO\Documents\Source\dl_bp_mimic_proyecto_integrador | cuda: True


In [2]:
from bp.experiment import get_data, load_label_stats, device
from bp.metrics import per_patient_metrics
from bp.paths import ARTIFACTS
y_mean, y_std = load_label_stats()
val = get_data('val', ['ppg'])
print("val:", val['X'].shape, "| y_mean", y_mean, "y_std", y_std)

val: torch.Size([9149, 1, 3750]) | y_mean [112.7766    58.479195] y_std [18.137848 12.511031]


## LB-0: la media de train

In [3]:
pred = np.tile(y_mean, (len(val['y']), 1))
lb0 = per_patient_metrics(pred, val['y'], val['patient'])
print({k: round(v, 3) for k, v in lb0.items() if k.startswith(('mae', 'me_', 'de_', 'pooled_sd'))})
(ARTIFACTS / 'lb0_val.json').write_text(json.dumps(lb0, indent=1))

{'mae_pas': 14.156, 'me_pas': -0.054, 'de_pas': 11.454, 'pooled_sd_pas': 17.456, 'mae_pad': 9.539, 'me_pad': -0.295, 'de_pad': 6.554, 'pooled_sd_pad': 12.066}


463

## Chequeos antes de entrenar (PPG y PPG+ECG)
Dropout desactivado en el chequeo de sobreajuste; la última capa parte en 0, así que la predicción inicial es la media.

In [4]:
from bp.train import sanity_checks, DEFAULT_CFG
dev = device()
results = {}
for inputs in (['ppg'], ['ppg', 'ecg']):
    tr = get_data('train', inputs)
    for builder in ('build_mlp', 'build_gru'):
        r = sanity_checks(builder, tr, tr['X'].shape[1], DEFAULT_CFG, dev)
        results[f"{builder}|{'+'.join(inputs)}"] = r
        print(f"{builder:10s} {'+'.join(inputs):8s}", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if k != 'grad_zero_layers'},
              "capas sin gradiente:", r['grad_zero_layers'])
        assert 0.8 <= r['init_mse'] <= 1.2, "MSE inicial fuera de 1 +- 0.2"
        assert r['overfit_loss'] < 0.05 and r['overfit_steps'] < 300, "no memoriza 64 ejemplos"
        assert r['grad_all_finite'] and not r['grad_zero_layers'], "gradientes no finitos o nulos"
        assert r['eval_deterministic'], "eval() no es determinista"
    del tr; torch.cuda.empty_cache()
(ARTIFACTS / 'sanity_checks.json').write_text(json.dumps(results, indent=1))
print("todos los chequeos pasan")

build_mlp  ppg      {'init_mse': 0.9449, 'grad_all_finite': True, 'grad_n_layers': 11, 'overfit_loss': 0.0447, 'overfit_steps': 28, 'eval_deterministic': True} capas sin gradiente: []


build_gru  ppg      {'init_mse': 0.9449, 'grad_all_finite': True, 'grad_n_layers': 10, 'overfit_loss': 0.049, 'overfit_steps': 199, 'eval_deterministic': True} capas sin gradiente: []


build_mlp  ppg+ecg  {'init_mse': 0.9449, 'grad_all_finite': True, 'grad_n_layers': 11, 'overfit_loss': 0.0491, 'overfit_steps': 27, 'eval_deterministic': True} capas sin gradiente: []


build_gru  ppg+ecg  {'init_mse': 0.9449, 'grad_all_finite': True, 'grad_n_layers': 10, 'overfit_loss': 0.0487, 'overfit_steps': 104, 'eval_deterministic': True} capas sin gradiente: []
todos los chequeos pasan
